# 💼 LinkedIn Jobs — Salary Dashboard
**Regions: USA · Canada · Africa**

> Place this notebook in the **same folder** as the three CSV files:
> `linkedin-jobs-usa.csv`, `linkedin-jobs-canada.csv`, `linkedin-jobs-africa.csv`

In [ ]:
# ── CELL 1 · IMPORTS ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')

In [ ]:
# ── CELL 2 · LOAD DATA ───────────────────────────────────────────────────────
frames = []
for region, fname in [('USA', 'linkedin-jobs-usa.csv'),
                      ('Canada', 'linkedin-jobs-canada.csv'),
                      ('Africa', 'linkedin-jobs-africa.csv')]:
    try:
        df = pd.read_csv(fname)
        df['Region'] = region
        frames.append(df)
        print(f'  {region}: {len(df):,} rows loaded')
    except FileNotFoundError:
        print(f'  WARNING: {fname} not found — skipping {region}')

all_df = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows: {len(all_df):,}')
print(f'Columns   : {list(all_df.columns)}')

In [ ]:
# ── CELL 3 · CLEANING & FEATURE ENGINEERING ──────────────────────────────────

# ① Dates
all_df['posted_date'] = pd.to_datetime(all_df['posted_date'], errors='coerce')

# ② Drop exact duplicates
before = len(all_df)
all_df = all_df.drop_duplicates()
print(f'Duplicates removed: {before - len(all_df)}')

# ③ Salary extraction  (handles the messy $x,xxx.xx\r\n - \r\n $y,yyy.yy format)
def extract_salaries(salary_str):
    if pd.isna(salary_str):
        return pd.Series([np.nan, np.nan])
    # Remove whitespace / newlines so regex works cleanly
    cleaned = re.sub(r'[\r\n\s]+', ' ', str(salary_str))
    # Extract numbers that follow a '$'
    numbers = re.findall(r'\$\s*([0-9,]+(?:\.\d+)?)', cleaned)
    # Fallback: grab any plain number
    if not numbers:
        numbers = re.findall(r'\b\d+(?:,\d{3})*(?:\.\d+)?\b', cleaned)
    numbers = [float(n.replace(',', '')) for n in numbers]
    if len(numbers) >= 2:
        return pd.Series([min(numbers[:2]), max(numbers[:2])])
    elif len(numbers) == 1:
        return pd.Series([numbers[0], numbers[0]])
    return pd.Series([np.nan, np.nan])

all_df[['min_salary', 'max_salary']] = all_df['salary'].apply(extract_salaries)

# Keep only annual-range values (> $30 k)  — drops hourly / monthly noise
all_df.loc[all_df['min_salary'] < 30_000, 'min_salary'] = np.nan
all_df.loc[all_df['max_salary'] < 30_000, 'max_salary'] = np.nan
all_df['avg_salary'] = (all_df['min_salary'] + all_df['max_salary']) / 2

valid_sal = all_df['avg_salary'].notna().sum()
print(f'Rows with valid salary: {valid_sal:,}  '
      f'({valid_sal / len(all_df) * 100:.1f}% of total)')

# ④ Parse criteria column  →  Seniority level / Employment type / Job function / Industries
def parse_criteria(criteria_str):
    try:
        items = ast.literal_eval(str(criteria_str))
        result = {}
        for item in items:
            result.update(item)
        return result
    except Exception:
        return {}

criteria_expanded = all_df['criteria'].dropna().apply(parse_criteria).apply(pd.Series)
all_df = pd.concat([all_df, criteria_expanded.reindex(all_df.index)], axis=1)
print(f'Criteria columns added: {list(criteria_expanded.columns)}')

# ⑤ Seniority from job title
def title_seniority(title):
    t = str(title).lower()
    if any(w in t for w in ['senior', 'sr.', ' sr ', 'lead', 'principal', 'manager', 'director', 'head']):
        return 'Senior / Lead'
    elif any(w in t for w in ['junior', 'jr.', ' jr ', 'entry', 'associate']):
        return 'Junior / Entry'
    return 'Mid-Level / Unspecified'

all_df['title_experience_level'] = all_df['title'].apply(title_seniority)

print('\nCleaning complete ✓')

In [ ]:
# ── CELL 4 · HELPER ──────────────────────────────────────────────────────────
def plot_bar(data, x, y, title, xlabel, ylabel,
             orient='v', figsize=(10, 5), palette='viridis', fmt=None):
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(x=x, y=y, data=data, palette=palette, ax=ax)
    for container in ax.containers:
        if fmt:
            ax.bar_label(container, fmt=fmt, padding=3)
        else:
            ax.bar_label(container, padding=3)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    plt.tight_layout()
    plt.show()

print('Helper function ready ✓')

In [ ]:
# ── CELL 5 · SALARY DISTRIBUTION ─────────────────────────────────────────────
salary_data = all_df['avg_salary'].dropna()

if salary_data.empty:
    print('No salary data available — skipping salary distribution plots.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    sns.histplot(salary_data, bins=30, kde=True, color='steelblue', ax=axes[0])
    axes[0].set_title('Distribution of Average Annual Salaries', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Average Salary ($)')
    axes[0].set_ylabel('Frequency')

    # Box plot
    box_data = all_df[['min_salary', 'max_salary', 'avg_salary']].dropna(how='all')
    sns.boxplot(data=box_data, orient='h', palette='Set2', ax=axes[1])
    axes[1].set_title('Salary Ranges — Min / Max / Avg', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Salary ($)')

    plt.tight_layout()
    plt.show()

    print(salary_data.describe().apply(lambda x: f'${x:,.0f}'))

In [ ]:
# ── CELL 6 · REGIONAL ANALYSIS (salary + job counts) ─────────────────────────

# 6-a  Job count per region
region_counts = all_df['Region'].value_counts().reset_index()
region_counts.columns = ['Region', 'Count']
plot_bar(region_counts, x='Region', y='Count',
         title='Job Postings by Region',
         xlabel='Region', ylabel='Number of Jobs', palette='Set2')

# 6-b  Average salary per region  (Africa has no salary data → shows only USA & Canada)
salary_by_region = (
    all_df.dropna(subset=['avg_salary'])
          .groupby('Region', as_index=False)
          .agg(avg_salary=('avg_salary', 'mean'),
               records=('avg_salary', 'count'))
          .sort_values('avg_salary', ascending=False)
)

if salary_by_region.empty:
    print('No salary data found for any region.')
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = sns.barplot(x='Region', y='avg_salary', data=salary_by_region,
                       palette='Set2', ax=ax)
    ax.bar_label(bars.containers[0], fmt='$%.0f', padding=4, fontsize=10)
    ax.set_title('Average Annual Salary by Region', fontsize=14, fontweight='bold')
    ax.set_xlabel('Region')
    ax.set_ylabel('Average Salary ($)')
    plt.tight_layout()
    plt.show()

    display(salary_by_region) if "display" in dir(__builtins__) else print(salary_by_region.to_string(index=False))

In [ ]:
# ── CELL 7 · SALARY vs WORK TYPE ─────────────────────────────────────────────
work_sal = all_df.dropna(subset=['avg_salary', 'onsite_remote'])

if work_sal.empty:
    print('No data for salary vs work-type plot.')
else:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x='onsite_remote', y='avg_salary', data=work_sal, palette='cool')
    plt.title('Average Salary by Workspace Setup', fontsize=14, fontweight='bold')
    plt.xlabel('Workspace Setup')
    plt.ylabel('Average Salary ($)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── CELL 8 · WORK TYPE COUNTS ────────────────────────────────────────────────
job_type = all_df['onsite_remote'].value_counts().reset_index()
job_type.columns = ['Job Type', 'Count']
plot_bar(job_type, x='Job Type', y='Count',
         title='Job Postings by Workspace Setup',
         xlabel='Setup', ylabel='Number of Jobs', palette='cool')

In [ ]:
# ── CELL 9 · TOP 15 HIRING COMPANIES ─────────────────────────────────────────
top_companies = all_df['company'].value_counts().head(15).reset_index()
top_companies.columns = ['Company', 'Job Count']
plot_bar(top_companies, x='Job Count', y='Company',
         title='Top 15 Companies Hiring',
         xlabel='Number of Postings', ylabel='Company',
         orient='h', figsize=(10, 8), palette='magma')

In [ ]:
# ── CELL 10 · TOP 10 LOCATIONS ────────────────────────────────────────────────
top_locations = all_df['location'].value_counts().head(10).reset_index()
top_locations.columns = ['Location', 'Job Count']
plot_bar(top_locations, x='Job Count', y='Location',
         title='Top 10 Locations for Job Postings',
         xlabel='Number of Postings', ylabel='Location',
         orient='h', figsize=(10, 6), palette='crest')

In [ ]:
# ── CELL 11 · BEST DAYS TO APPLY ─────────────────────────────────────────────
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
days_counts = (
    all_df['posted_date'].dt.day_name()
                         .value_counts()
                         .reindex(days_order)
                         .reset_index()
)
days_counts.columns = ['Day of Week', 'Postings']
plot_bar(days_counts, x='Day of Week', y='Postings',
         title='Job Postings by Day of the Week',
         xlabel='Day', ylabel='Number of Postings', palette='autumn')

In [ ]:
# ── CELL 12 · EXPERIENCE LEVEL FROM TITLE ────────────────────────────────────
exp_levels = all_df['title_experience_level'].value_counts().reset_index()
exp_levels.columns = ['Experience Level', 'Count']
plot_bar(exp_levels, x='Experience Level', y='Count',
         title='Experience Levels Inferred from Job Titles',
         xlabel='Level', ylabel='Count', palette='pastel')

In [ ]:
# ── CELL 13 · CRITERIA BREAKDOWN ─────────────────────────────────────────────
criteria_cols = {
    'Seniority level': ('Jobs by Seniority Level', 'Blues_r'),
    'Employment type': ('Jobs by Employment Type',  'Greens_r'),
    'Job function':    ('Top 10 Job Functions',     'Oranges_r'),
    'Industries':      ('Top 10 Hiring Industries', 'Purples_r'),
}

for col, (title, palette) in criteria_cols.items():
    if col in all_df.columns and all_df[col].notna().any():
        data = all_df[col].value_counts().head(10).reset_index()
        data.columns = [col, 'Count']
        plot_bar(data, x='Count', y=col, title=title,
                 xlabel='Count', ylabel=col,
                 orient='h', figsize=(10, 6), palette=palette)
    else:
        print(f'Column "{col}" not found or empty — skipping.')

In [ ]:
# ── CELL 14 · SALARY HEATMAP — REGION × SENIORITY ────────────────────────────
sal_heat = (
    all_df.dropna(subset=['avg_salary', 'Seniority level'])
          .groupby(['Region', 'Seniority level'])['avg_salary']
          .mean()
          .unstack()
)

if sal_heat.empty:
    print('Not enough data for heatmap.')
else:
    plt.figure(figsize=(12, 4))
    sns.heatmap(sal_heat, annot=True, fmt=',.0f', cmap='YlOrRd',
                linewidths=0.5, linecolor='white')
    plt.title('Average Salary ($) — Region × Seniority Level',
              fontsize=14, fontweight='bold')
    plt.ylabel('Region')
    plt.xlabel('Seniority Level')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── CELL 15 · SUMMARY TABLE ───────────────────────────────────────────────────
summary = all_df.groupby('Region').agg(
    Total_Jobs        = ('title',      'count'),
    Jobs_With_Salary  = ('avg_salary', 'count'),
    Avg_Salary        = ('avg_salary', 'mean'),
    Median_Salary     = ('avg_salary', 'median'),
    Min_Salary        = ('min_salary', 'min'),
    Max_Salary        = ('max_salary', 'max'),
).reset_index()

for col in ['Avg_Salary', 'Median_Salary', 'Min_Salary', 'Max_Salary']:
    summary[col] = summary[col].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A')

display(summary) if "display" in dir(__builtins__) else print(summary.to_string(index=False))